# 02 - Removing the gender markers from the biographies

Both studies ask about wording, not about explicit mentions of gender. This
notebook removes the gender markers and the first names from every biography
and writes the cleaned text that all features are counted on.

- **Question:** Can the explicit marks of gender be taken off the surface of
  the text without touching the rest of the wording?
- **Expectation:** none directional; the notebook prepares the text.
- **Status:** second step of the series. The two functions are exported to
  `src/preprocessing.py`, so that the notebooks and Study 2 share one
  implementation.
- **Unit and test:** biography; share of texts that still hold a marker, the
  length before and after, a leakage test, and examples read by hand.
- **Inputs:** `outputs/df_clean.parquet` from notebook 01.
- **Outputs:** `outputs/df_preprocessed.parquet`, the columns of the input plus
  `text_clean` (gender markers and first names removed).
- **Run:** from a fresh kernel, top to bottom, from the folder `notebooks/`;
  the paths are relative to it. The name removal loads spaCy `en_core_web_lg`.
- **Decisions:** the third-person pronouns and the titles (`he`, `she`, `her`,
  `his`, `him`, `hers`, `himself`, `herself`, `mr`, `mrs`, `ms`) are removed by
  a regular expression and the first names by the spaCy named-entity
  recognition. First-person and second-person pronouns stay in the text; they
  are the subject of decision D1 of Study 2 and of section 5 of notebook 04b.

Sections: (1) data loading, (2) gender marker scrubbing, (3) first-name
removal, (4) full-dataset application, (5) validation A, residual markers,
(6) validation B, gender leakage, (7) export to `src/preprocessing.py`,
(8) save, (9) the versions of the environment.


## 1. Data Loading

Reads the cleaned dataset produced by `01_eda.ipynb`. The parquet file contains 393,977 bios across 27 occupations after EDA-stage filtering: exact duplicates removed, the `rapper` occupation excluded due to near-zero female representation (< 2%). Gender is encoded as a binary label (`Male` / `Female`) derived from the original dataset's `gender` field.

In [10]:
import os, pathlib, sys

# -- Working directory -> project root ---------------------------------------

# The notebook may be opened from notebooks/, from the project root or from a
# copy in notebooks/archive/. The root is the nearest folder upwards that holds
# both outputs/ and src/, so every path below can be written relative to it.
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / 'outputs').is_dir() and (p / 'src').is_dir()), None)
if ROOT is None:
    raise RuntimeError('project root not found: no folder at or above the '
                       'working directory holds both outputs/ and src/')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f'working directory: {ROOT.name}')

import pandas as pd
import numpy as np
import re

df = pd.read_parquet('outputs/df_clean.parquet')

print(f'Loaded: {len(df):,} bios | {df["occupation"].nunique()} occupations')
print(df['gender_label'].value_counts())
print()
print('Columns:', df.columns.tolist())
print('Sample bio:')
print(' ', df['hard_text'].iloc[0])

working directory: Thesis_Model
Loaded: 393,977 bios | 27 occupations
gender_label
Male      211799
Female    182178
Name: count, dtype: int64

Columns: ['hard_text', 'profession', 'gender', 'occupation', 'gender_label', 'word_count']
Sample bio:
  He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates


## 2. Gender Marker Scrubbing

Removes explicit gender markers following the exact scrubbing list from De-Arteaga et al. (2019): personal pronouns (`he`, `she`, `her`, `his`, `him`, `hers`, `himself`, `herself`) and gendered titles (`mr`, `mrs`, `ms`). Gaps are collapsed to a single space; no placeholder token (e.g. `[PRONOUN]`) is introduced, consistent with the original paper and with how spaCy's dependency parser handles token-level relationships (Honnibal & Montani, 2017).

Gendered nouns (`man`, `woman`, `men`, `women`) are **intentionally kept** - they carry occupational meaning rather than referencing the subject and are not part of the De-Arteaga list. The residual check below confirms 0.0000% remaining markers after scrubbing.

In [11]:
# Exact De-Arteaga et al. (2019) scrubbing list
_MARKER_RE = re.compile(
    r'\b(?:he|she|her|his|him|hers|himself|herself|mr|mrs|ms)\b',
    re.IGNORECASE
)

def remove_gender_markers(text: str) -> str:
    """
    Remove explicit gender markers following De-Arteaga et al. (2019).
    Gendered nouns (man, woman, men, women, boy, girl) are kept — content words.
    Gaps collapsed to single space.
    """
    text = _MARKER_RE.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

# --- Spot-check ---
print('=== Marker removal spot-check ===\n')
checks = [
    df[df['hard_text'].str.contains('HPV infections', na=False)].iloc[0],
    df[df['gender_label'] == 'Male'].sample(1, random_state=1).iloc[0],
    df[df['gender_label'] == 'Female'].sample(1, random_state=1).iloc[0],
]
for row in checks:
    orig    = row['hard_text']
    cleaned = remove_gender_markers(orig)
    print(f'[{row["occupation"]} | {row["gender_label"]}]')
    print(f'  BEFORE: {orig[:220]}')
    print(f'  AFTER:  {cleaned[:220]}')
    print()

# --- Residual check ---
residual = df['hard_text'].apply(remove_gender_markers).str.contains(
    _MARKER_RE
).mean() * 100
print(f'Residual markers after scrubbing: {residual:.4f}%  (target: 0%)')

=== Marker removal spot-check ===

[professor | Male]
  BEFORE: His areas of clinical practice include HIV medicine, sexually transmitted diseases (STDs), hospital acquired infections and tropical medicine. Dr. Castro’s research interests include HIV drug resistance, barrier methods 
  AFTER:  areas of clinical practice include HIV medicine, sexually transmitted diseases (STDs), hospital acquired infections and tropical medicine. Dr. Castro’s research interests include HIV drug resistance, barrier methods to p

[physician | Male]
  BEFORE: Dr. Park has indicated that his clinical interests include hypertension (high blood pressure) and kidney problems. His hospital/clinic affiliations include St. John's Hospital and St. John Providence Health System. AARP,
  AFTER:  Dr. Park has indicated that clinical interests include hypertension (high blood pressure) and kidney problems. hospital/clinic affiliations include St. John's Hospital and St. John Providence Health System. AARP, Cofinit

[

## 3. First-Name Removal

De-Arteaga et al. (2019) also remove first names to prevent the classifier from learning gender associations via name frequency patterns. A two-stage pipeline is used here:

1. **spaCy NER** (`en_core_web_lg`, NER component only) identifies `PERSON` entities. This avoids false positives from content words that a simple regex over capitalised tokens would misclassify (e.g. *General*, *Christian*, *Dr.*).
2. **gender_guesser** confirms that the entity's first token is gender-indicative (`male` / `female` / `mostly_male` / `mostly_female`). Names classified as `andy` (androgynous) or `unknown` are left in place - a conservative choice that reduces over-removal at the cost of slightly lower coverage.

The NER model is loaded with non-NER components disabled to reduce memory and runtime. The full pipeline (with tagger, parser, etc.) is loaded in `03_feature_engineering.ipynb`.

**Coverage on 500-bio sample:** ~38% of bios had at least one gender-indicative name removed, balanced across Male (37.6%) and Female (38.9%). The small asymmetry is expected and does not introduce directional bias.

*Limitation:* gender_guesser's name database is predominantly Western; coverage of non-Western names is lower and treated conservatively (kept). This is documented as a thesis limitation.

In [12]:
import spacy
import gender_guesser.detector as gg
import subprocess

try:
    spacy.load('en_core_web_lg')
except OSError:
    print('Downloading en_core_web_lg (~560 MB)...')
    subprocess.check_call(['python', '-m', 'spacy', 'download', 'en_core_web_lg'])

print('Loading spaCy en_core_web_lg (NER only)...')
nlp_ner = spacy.load('en_core_web_lg',
                     disable=['tok2vec', 'tagger', 'parser',
                              'attribute_ruler', 'lemmatizer'])
_guesser = gg.Detector()
print('Done.')

def _apply_name_removal(doc, text: str) -> str:
    """Core logic: given a pre-computed spaCy doc, remove gender-indicative PERSON names."""
    to_remove = []
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            first_token = ent.text.split()[0]
            g = _guesser.get_gender(first_token)
            if g in ('male', 'female', 'mostly_male', 'mostly_female'):
                to_remove.append(re.escape(ent.text))
    if not to_remove:
        return text
    pattern = re.compile('|'.join(to_remove), re.IGNORECASE)
    text = pattern.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

def remove_person_names(text: str) -> str:
    """Single-text wrapper around _apply_name_removal (used for spot-checks)."""
    return _apply_name_removal(nlp_ner(text), text)

# --- Coverage on sample (n=500 - full run in Cell 4 via nlp.pipe) ---
print('Running NER on 500-bio sample...')
sample = df.sample(500, random_state=42).copy()
sample['after_markers'] = sample['hard_text'].apply(remove_gender_markers)
sample['after_names']   = sample['after_markers'].apply(remove_person_names)
sample['name_removed']  = sample['after_markers'] != sample['after_names']

print(f'\n=== First-name removal coverage (n=500) ===')
print(f'  Names removed: {sample["name_removed"].sum()} ({sample["name_removed"].mean()*100:.1f}%)')
print('\nBy gender label:')
print(sample.groupby('gender_label')['name_removed'].mean()
      .apply(lambda x: f'{x*100:.1f}%'))

print('\n=== Spot-check (name removed) ===')
for _, row in sample[sample['name_removed']].sample(3, random_state=7).iterrows():
    print(f'[{row["occupation"]} | {row["gender_label"]}]')
    print(f'  after markers: {row["after_markers"][:180]}')
    print(f'  after names:   {row["after_names"][:180]}')
    print()

print('=== Spot-check (PERSON found but kept — unknown/andy) ===')
kept = 0
for text in sample[~sample['name_removed']]['after_markers']:
    doc = nlp_ner(text)
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            g = _guesser.get_gender(ent.text.split()[0])
            if g not in ('male', 'female', 'mostly_male', 'mostly_female'):
                print(f'  "{ent.text}" → {g}')
                kept += 1
                break
    if kept >= 2:
        break

Loading spaCy en_core_web_lg (NER only)...
Done.
Running NER on 500-bio sample...

=== First-name removal coverage (n=500) ===
  Names removed: 191 (38.2%)

By gender label:
gender_label
Female    38.9%
Male      37.6%
Name: name_removed, dtype: object

=== Spot-check (name removed) ===
[professor | Male]
  after markers: Before came to Tampere was a Senior Lecturer (Associate Professor) at the Queen’s University Belfast at the Center for Cancer Research and Cell Biology. received training as a Seni
  after names:   Before came to Tampere was a Senior Lecturer (Associate Professor) at the Queen’s University Belfast at the Center for Cancer Research and Cell Biology. received training as a Seni

[poet | Female]
  after markers: is also an editor of Poetry Wales. approach has always been transcultural and has collaborated and curated internationally; poetry has been translated into Arabic, Croatian, Czech,
  after names:   is also an editor of Poetry Wales. approach has always been trans

## 4. Full-Dataset Application

Applies the two-stage pipeline to all 393,977 bios. Gender marker removal runs in ~40 s (pure regex). NER-based name removal uses `nlp.pipe(batch_size=512)` for throughput and takes ~40 min on CPU. The cleaned text is stored in `df['text_clean']`.

In [13]:
import time

# Step 1: marker removal (fast - no NER)
print('Step 1: removing gender markers...')
t0 = time.time()
texts_after_markers = df['hard_text'].apply(remove_gender_markers).tolist()
print(f'  Done in {time.time()-t0:.1f}s')

# Step 2: batch NER name removal via nlp.pipe()
print(f'\nStep 2: batch NER name removal ({len(texts_after_markers):,} bios, batch_size=512)...')
print('  (expected: ~10-20 min)')
t0 = time.time()

text_clean = []
total = len(texts_after_markers)

for i, (doc, text) in enumerate(
    zip(nlp_ner.pipe(texts_after_markers, batch_size=512), texts_after_markers)
):
    text_clean.append(_apply_name_removal(doc, text))
    if (i + 1) % 50_000 == 0:
        elapsed = time.time() - t0
        pct = (i + 1) / total * 100
        print(f'  {i+1:,}/{total:,} ({pct:.0f}%)  —  {elapsed/60:.1f} min elapsed')

elapsed = time.time() - t0
print(f'  Done in {elapsed/60:.1f} min')

df['text_clean'] = text_clean

# --- Before/after spot-check on 3 bios ---
print('\n=== Before / After spot-check ===\n')
idxs = [
    df[df['gender_label'] == 'Female'].sample(1, random_state=3).index[0],
    df[df['gender_label'] == 'Male'].sample(1, random_state=3).index[0],
    df[df['hard_text'].str.contains('Anissa', na=False)].index[0],
]
for idx in idxs:
    row = df.loc[idx]
    print(f'[{row["occupation"]} | {row["gender_label"]}]')
    print(f'  ORIGINAL:   {row["hard_text"][:220]}')
    print(f'  CLEAN:      {row["text_clean"][:220]}')
    print()

print(f'text_clean column added. Non-null: {df["text_clean"].notna().sum():,}')

Step 1: removing gender markers...
  Done in 20.9s

Step 2: batch NER name removal (393,977 bios, batch_size=512)...
  (expected: ~10-20 min)
  50,000/393,977 (13%)  —  2.9 min elapsed
  100,000/393,977 (25%)  —  5.4 min elapsed
  150,000/393,977 (38%)  —  7.9 min elapsed
  200,000/393,977 (51%)  —  11.9 min elapsed
  250,000/393,977 (63%)  —  14.8 min elapsed
  300,000/393,977 (76%)  —  18.6 min elapsed
  350,000/393,977 (89%)  —  22.6 min elapsed
  Done in 25.8 min

=== Before / After spot-check ===

[professor | Female]
  ORIGINAL:   She teaches Victorian and early twentieth-century British literature, with special attention to the novel, the fin de siècle, and the intersection of literature and the visual arts. Her scholarly work has focused on the 
  CLEAN:      teaches Victorian and early twentieth-century British literature, with special attention to the novel, the fin de siècle, and the intersection of literature and the visual arts. scholarly work has focused on the work of 



**Spot-check interpretation:** Pronouns (`She`, `Her`, `His`, `He`) are correctly removed across all examples. The nurse bio confirms multi-token name removal: `Anissa L Sonntag` is identified as a single `PERSON` entity, classified as female via gender_guesser, and removed in full. `Dr. Pannozzo` is retained - `Dr.` is not in the De-Arteaga scrubbing list, and `Pannozzo` as a last name is classified as `unknown` by gender_guesser, consistent with the conservative two-stage design.

Validation of `text_clean` (residual marker check and logistic regression leakage test) follows in Sections 5 and 6.

## 5. Validation A - Residual Marker Check

Scans every `text_clean` bio for any remaining gender markers from the De-Arteaga list. A 0.00% residual confirms that the scrubbing step is complete and symmetric across gender groups. The total-marker count compares the original and cleaned corpora to quantify the volume of signal removed.

In [14]:
print('=== Validation A: Residual Gender Markers in text_clean ===\n')

# Use pattern string (non-capturing) to avoid pandas capture-group warning
_MARKER_PAT = r'\b(?:he|she|her|his|him|hers|himself|herself|mr|mrs|ms)\b'

# --- Residual count ---
residual_mask = df['text_clean'].str.contains(_MARKER_PAT, case=False, regex=True, na=False)
residual_pct  = residual_mask.mean() * 100
n_residual    = residual_mask.sum()

print(f'Bios with residual markers : {n_residual:,}  ({residual_pct:.4f}%)')
print(f'Threshold set in advance: 0.01%  |  measured: {residual_pct:.4f}%')

# --- By gender ---
print('\nResidual by gender:')
print(df.groupby('gender_label')['text_clean']
      .apply(lambda s: s.str.contains(_MARKER_PAT, case=False, regex=True, na=False).mean() * 100)
      .apply(lambda x: f'{x:.4f}%'))

# --- Total markers removed ---
def count_markers(text):
    return len(_MARKER_RE.findall(text)) if isinstance(text, str) else 0

markers_before = df['hard_text'].apply(count_markers).sum()
markers_after  = df['text_clean'].apply(count_markers).sum()

print(f'\nTotal marker tokens in hard_text : {markers_before:,}')
print(f'Total marker tokens in text_clean: {markers_after:,}')
print(f'Markers removed                  : {markers_before - markers_after:,}  '
      f'({(markers_before - markers_after)/markers_before*100:.1f}%)')

# --- Show all residual cases ---
if n_residual > 0:
    print(f'\nAll {n_residual} bios with residual markers:')
    for _, row in df[residual_mask].iterrows():
        found = set(m.lower() for m in _MARKER_RE.findall(row['text_clean']))
        print(f'  [{row["occupation"]} | {row["gender_label"]}]  remaining: {found}')
        print(f'  {row["text_clean"][:300]}')

=== Validation A: Residual Gender Markers in text_clean ===

Bios with residual markers : 2  (0.0005%)
Threshold set in advance: 0.01%  |  measured: 0.0005%

Residual by gender:
gender_label
Female    0.0005%
Male      0.0005%
Name: text_clean, dtype: object

Total marker tokens in hard_text : 1,196,248
Total marker tokens in text_clean: 2
Markers removed                  : 1,196,246  (100.0%)

All 2 bios with residual markers:
  [professor | Female]  remaining: {'ms'}
  Dr. attend Norfolk State University (NSU), where graduat with a Bachelor of Arts in Political Science. attend graduate school at Touro College in New York City and earn a MS with dual certification in general ucation and special ucation. Additionally, attend Cambridge College where receiv an .S in A
  [professor | Male]  remaining: {'mr'}
  has directed first-year composition, served as an ePortfolio faculty fellow, and delivered numerous workshops and consultations on electronic portfolio initiatives. has taught cours

**Result:** The residual matches printed above are false positives rather than
gendered titles: they carry `MS` as an academic degree (*Master of Science*), or
`Mr.` as a title referring to a third person mentioned in the text. A
word-boundary regular expression cannot tell `Ms.`/`Mr.` (a title) from
`MS`/`MR` (a degree or a reference) without sentence-level context. The rate is
printed above, for the corpus and per gender group; it stays far below the
threshold fixed in advance and is the same on both sides, so these cases are
treated as noise and kept as a documented limitation. The number of marker
tokens before and after the scrubbing is printed above as well.


## 6. Validation B - Gender Leakage Test

A logistic regression on TF-IDF features is used to measure how much gender signal remains in `text_clean` after scrubbing. This serves two purposes:

1. **Scrubbing sanity check:** If accuracy after scrubbing were similar to `hard_text`, explicit markers would not have been the dominant signal - or scrubbing would have failed. A meaningful drop in AUC confirms markers were informative and were successfully removed.
2. **Thesis framing:** Residual predictability above chance (~53.8% majority baseline) is expected and meaningful. Professional bios contain occupation-correlated vocabulary (e.g. *nursing*, *software*, *classroom*) that reflects structural gender segregation - not a scrubbing failure but the core phenomenon under study.

Both `hard_text` and `text_clean` are evaluated on an 80/20 stratified split to make the before/after comparison explicit.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

y = (df['gender_label'] == 'Female').astype(int)  # 1=Female, 0=Male
majority_baseline = max(y.mean(), 1 - y.mean())

def eval_gender_leakage(texts, y, label):
    X_tr, X_te, y_tr, y_te = train_test_split(
        texts, y, test_size=0.2, random_state=42, stratify=y
    )
    tfidf = TfidfVectorizer(max_features=50_000, min_df=5, ngram_range=(1, 2))
    X_tr_tf = tfidf.fit_transform(X_tr)
    X_te_tf = tfidf.transform(X_te)

    lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
    lr.fit(X_tr_tf, y_tr)

    acc = accuracy_score(y_te, lr.predict(X_te_tf))
    auc = roc_auc_score(y_te, lr.predict_proba(X_te_tf)[:, 1])
    print(f'{label}')
    print(f'  Accuracy : {acc*100:.1f}%  (majority baseline: {majority_baseline*100:.1f}%)')
    print(f'  ROC-AUC  : {auc:.3f}')
    return lr, tfidf, auc

print('=== Validation B: Gender Leakage Test ===\n')
lr_raw,   tfidf_raw,   auc_raw   = eval_gender_leakage(df['hard_text'].fillna(''), y, 'hard_text  (before scrubbing)')
print()
lr_clean, tfidf_clean, auc_clean = eval_gender_leakage(df['text_clean'].fillna(''), y, 'text_clean (after scrubbing)')

print(f'\nAUC reduction from scrubbing: {auc_raw:.3f} → {auc_clean:.3f}  (Δ = {auc_raw - auc_clean:.3f})')

# --- Top features from text_clean model ---
print('\n=== Top 15 features predicting Female (text_clean) ===')
feat_names = np.array(tfidf_clean.get_feature_names_out())
coefs      = lr_clean.coef_[0]
top_female = feat_names[np.argsort(coefs)[-15:][::-1]]
top_male   = feat_names[np.argsort(coefs)[:15]]
print(f'  Female → {list(top_female)}')
print(f'  Male   → {list(top_male)}')

=== Validation B: Gender Leakage Test ===

hard_text  (before scrubbing)
  Accuracy : 99.8%  (majority baseline: 53.8%)
  ROC-AUC  : 1.000

text_clean (after scrubbing)
  Accuracy : 72.1%  (majority baseline: 53.8%)
  ROC-AUC  : 0.794

AUC reduction from scrubbing: 1.000 → 0.794  (Δ = 0.206)

=== Top 15 features predicting Female (text_clean) ===
  Female → ['women', 'husband', 'feminist', 'woman', 'female', 'with husband', 'nurse', 'girl', 'practices medicine', 'gender', 'mother of', 'nursing', 'mother', 'miss', 'and husband']
  Male   → ['wife', 'with wife', 'and wife', 'father of', 'justin', 'father', 'football', 'man', 'chairman', 'men', 'gay', 'baseball', 'wife and', 'sports', 'fishing']


**Result:** The scrubbing lowers how well a classifier can tell the gender of a
biography, from the value on `hard_text` to the value on `text_clean` printed
above. The near-perfect AUC before the scrubbing reflects the near-universal
presence of gendered pronouns and is expected rather than suspicious.

What remains is **implicit gender signal** in the content of the biographies,
not a failure of the scrubbing. The top predictive features show three
structural sources:

- **Occupational segregation:** `nurse`/`nursing` (female) against
  `football`/`baseball`/`chairman` (male) - gendered professional domains.
- **Relational self-presentation:** `husband`/`mother of` (female) against
  `wife`/`father of` (male) - women and men refer to family roles in
  systematically different ways inside a professional biography.
- **Identity language:** `feminist`/`gender`/`female` (female) - women use
  explicit terms of gender identity more often.

This residual predictability is the phenomenon the thesis studies: gender in
professional biographies works through content and framing, not only through
explicit markers. The scrubbing isolates that implicit dimension for the
notebooks that follow.

*Note:* `miss` appears among the top female predictors but is absent from the
De-Arteaga scrubbing list, because it is also a common verb. Removing it would
take content-bearing tokens with it. This is a limitation of scrubbing at the
level of tokens, without syntactic context.


## 7. Export to `src/preprocessing.py`

The finalized scrubbing functions are written to [src/preprocessing.py](../src/preprocessing.py) so downstream notebooks can import them without redefining. Key changes from the inline notebook versions: `nlp_ner` and `guesser` are explicit parameters (not closures), and a `batch_remove_person_names()` helper wraps the `nlp.pipe()` pattern. The cell below verifies the import and runs a smoke test.

In [16]:
from src.preprocessing import (
    remove_gender_markers   as src_remove_markers,
    load_ner                as src_load_ner,
    remove_person_names     as src_remove_names,
    batch_remove_person_names,
)

print('src.preprocessing imported successfully.')

# Smoke test: verify src functions produce identical output to inline versions
nlp_ner_src, guesser_src = src_load_ner()

test_bio = df['hard_text'].iloc[0]

inline_result = remove_person_names(
    remove_gender_markers(test_bio)
)
src_result = src_remove_names(
    src_remove_markers(test_bio), nlp_ner_src, guesser_src
)

assert inline_result == src_result, 'MISMATCH between inline and src output!'
print('Smoke test passed — src output matches inline output.')
print(f'\nOriginal : {test_bio[:180]}')
print(f'Cleaned  : {src_result[:180]}')

src.preprocessing imported successfully.
Smoke test passed — src output matches inline output.

Original : He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michig
Cleaned  : is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." earned a master’s degree in computer science from the University of Michigan-Dea


## 8. Save Preprocessed Dataset

Writes the final dataset to `outputs/df_preprocessed.parquet`. This file is the single input for all downstream notebooks (`03_feature_engineering`, `04_study1_analysis`, `05_study2_experiment`). The original `hard_text` column is retained alongside `text_clean` for reference and diagnostic purposes.

In [17]:
out_path = 'outputs/df_preprocessed.parquet'

cols = ['hard_text', 'text_clean', 'profession', 'occupation', 'gender', 'gender_label', 'word_count']
df[cols].to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1_048_576

print(f'Saved: {out_path}')
print(f'Shape: {df[cols].shape}')
print(f'Size : {size_mb:.1f} MB')
print(f'\nColumns saved:')
for col in cols:
    print(f'  {col:<15} {str(df[col].dtype):<12} '
          f'non-null: {df[col].notna().sum():,}')

Saved: outputs/df_preprocessed.parquet
Shape: (393977, 7)
Size : 181.8 MB

Columns saved:
  hard_text       object       non-null: 393,977
  text_clean      object       non-null: 393,977
  profession      int64        non-null: 393,977
  occupation      object       non-null: 393,977
  gender          int64        non-null: 393,977
  gender_label    object       non-null: 393,977
  word_count      int64        non-null: 393,977


## 9. Versions of the environment

The counts of this notebook and of notebook 03 depend on the tokenisation, so
the versions behind them are part of the result. This section reads the
installed versions and the date of the downloaded Kaggle file; it loads no model
and changes nothing. The values go into `environment/requirements.txt`, the
README and the appendix of the thesis.


In [18]:
# -- 9. Versions of the environment ------------------------------------------

# Read from the installed distributions, so the list cannot drift from the
# environment that produced the results.
import datetime
import os
import platform
from importlib.metadata import PackageNotFoundError, version

PACKAGES = [
    "spacy", "en_core_web_lg", "gender-guesser", "scikit-learn",
    "pandas", "numpy", "pyarrow", "scipy", "statsmodels",
    "matplotlib", "seaborn",
    "sentence-transformers", "transformers", "torch",
]

print(f"python                 {platform.python_version()}")
for name in PACKAGES:
    found = None
    for candidate in (name, name.replace("_", "-"), name.replace("-", "_")):
        try:
            found = version(candidate)
            break
        except PackageNotFoundError:
            continue
    print(f"{name:<22} {found or 'not installed'}")

# The job advertisements are not in the repository; the date below is the last
# change of the downloaded file on disk, which is its download or unzip date.
JOB_FILE = "data/job_descriptions/job_postings.csv"
if os.path.exists(JOB_FILE):
    changed = datetime.datetime.fromtimestamp(os.path.getmtime(JOB_FILE))
    print(f"\njob_postings.csv last changed on {changed:%Y-%m-%d}")
else:
    print(f"\n{JOB_FILE} is not in place; the Kaggle file is downloaded separately")


python                 3.13.9
spacy                  3.8.14
en_core_web_lg         3.8.0
gender-guesser         0.4.0
scikit-learn           1.6.1
pandas                 2.2.3
numpy                  2.1.3
pyarrow                19.0.0
scipy                  1.15.3
statsmodels            0.14.4
matplotlib             3.10.0
seaborn                0.13.2
sentence-transformers  6.0.1
transformers           5.16.1
torch                  2.14.0

job_postings.csv last changed on 2026-08-31
